In [1]:
# model_selection.py
# -*- coding: utf-8 -*-
"""
功能：
1. 读取广告原始数据
2. 特征工程（去掉无关字段，构造 X, y）
3. 比较 LightGBM / XGBoost / RandomForest
4. 自动选择最优模型并训练
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

# ========== RMSE scorer ==========
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

# ========== 模型比较函数 ==========
def compare_and_select_model(X, y):
    models = {
        "LightGBM": lgb.LGBMRegressor(random_state=42),
        "XGBoost": XGBRegressor(random_state=42, verbosity=0),
        "RandomForest": RandomForestRegressor(random_state=42)
    }

    results = {}
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for name, model in models.items():
        rmse_scores = cross_val_score(model, X, y, cv=kf, scoring=rmse_scorer)
        rmse_mean = -np.mean(rmse_scores)

        r2_scores = cross_val_score(model, X, y, cv=kf, scoring="r2")
        r2_mean = np.mean(r2_scores)

        results[name] = {"RMSE": rmse_mean, "R2": r2_mean}
        print(f"{name}: RMSE={rmse_mean:.4f}, R2={r2_mean:.4f}")

    # 选 RMSE 最小的模型
    best_model_name = min(results, key=lambda k: results[k]["RMSE"])
    print(f"\n✅ 最优模型: {best_model_name} "
          f"(RMSE={results[best_model_name]['RMSE']:.4f}, "
          f"R2={results[best_model_name]['R2']:.4f})")

    best_model = models[best_model_name]
    best_model.fit(X, y)

    return best_model, results

# ========== 主程序 ==========
def main():
    # 1. 读取数据
    file_path = "/Users/qiulianpeng/比赛/赛题数据(天)_组别211.xlsx"
    df = pd.read_excel(file_path)
    df.columns = df.columns.str.strip()

    print("原始数据大小:", df.shape)

    # 2. 设置目标变量 y
    y = df["业绩"]   # 也可以改成 "订单"

    # 3. 构造特征 X（去掉非数值型 ID）
    drop_cols = ["业绩", "订单", "活动", "渠道", "广告系列ID_h", "广告组ID_h", "date"]
    X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

    print("特征维度:", X.shape)

    # 4. 模型比较并选择最优
    best_model, results = compare_and_select_model(X, y)

    # 5. 保存结果
    pd.DataFrame(results).T.to_excel("model_comparison_results.xlsx", index=True)
    print("📂 模型比较结果已保存到 model_comparison_results.xlsx")

if __name__ == "__main__":
    main()


原始数据大小: (4224, 10)
特征维度: (4224, 4)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000250 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 772
[LightGBM] [Info] Number of data points in the train set: 3379, number of used features: 4
[LightGBM] [Info] Start training from score 6534.602713
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000200 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 772
[LightGBM] [Info] Number of data points in the train set: 3379, number of used features: 4
[LightGBM] [Info] Start training from score 6743.881461
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000269 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 772
[LightGBM] [Info] Number of data points in the train set: 3379, number of used feature

In [4]:
# -*- coding: utf-8 -*-
"""
完整广告预算分配脚本（修正 day-level 约束，含多臂老虎机探索）
- 路径：'/Users/qiulianpeng/比赛/赛题数据(天)_组别211.xlsx'
- 输出：'/Users/qiulianpeng/比赛/12月广告最终安排_fixed.xlsx'
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from pulp import LpProblem, LpVariable, lpSum, LpMaximize, LpBinary
from itertools import product
import warnings
warnings.filterwarnings("ignore")

# ---------------- 参数 ----------------
FILE_PATH = '/Users/qiulianpeng/比赛/赛题数据(天)_组别211.xlsx'
MONTH = '2026-12'
HOLIDAYS = ['2026-12-20', '2026-12-22', '2026-12-24', '2026-12-25']
HOLIDAY_MULTIPLIER = 1            # 节假日效益倍数
TOTAL_BUDGET = 1_000_000            # 总预算
MAX_DAYS_PER_ACTIVITY = 6           # 每个活动最多投放天数（不同日期数）
EXPLORATION_STEP = 0.1              # 探索扰动幅度（±）
MAX_CANDIDATES = 3000               # 可选：若候选过多，设置为 int 保留 top-K（按预测订单）；None 表示不裁剪

OUTPUT_PATH = '/Users/qiulianpeng/比赛/12月广告最终安排_fixed-无节日效应版2.xlsx'

# ---------------- 1. 读取历史数据 ----------------
df = pd.read_excel(FILE_PATH)
df.columns = df.columns.str.strip()

required_cols = ['活动','活动第几天','渠道','广告系列ID_h','广告组ID_h','业绩','订单','花费','曝光','点击']
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"历史数据缺少列：{c}")

num_cols = ['业绩','订单','花费','曝光','点击']
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# ---------------- 2. 准备训练特征 ----------------
train_features = ['渠道','广告系列ID_h','广告组ID_h','花费','曝光','点击','活动第几天']
target = '订单'

for c in ['渠道','广告系列ID_h','广告组ID_h']:
    df[c] = df[c].astype('category')

X = df[train_features]
y = df[target].astype(float)

X_enc = pd.get_dummies(X, columns=['渠道','广告系列ID_h','广告组ID_h'], drop_first=False).astype(float)
X_train, X_val, y_train, y_val = train_test_split(X_enc, y, test_size=0.2, random_state=42)

# 使用随机森林替换 LightGBM
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# ---------------- 3. 生成候选组合 ----------------
activities = df['活动'].unique()
channels = df['渠道'].cat.categories if hasattr(df['渠道'], 'cat') else df['渠道'].unique()
series = df['广告系列ID_h'].cat.categories if hasattr(df['广告系列ID_h'], 'cat') else df['广告系列ID_h'].unique()
groups = df['广告组ID_h'].cat.categories if hasattr(df['广告组ID_h'], 'cat') else df['广告组ID_h'].unique()
days = list(range(1, 32))

grp_mean = df.groupby(['活动','渠道','广告系列ID_h','广告组ID_h'])[['花费','曝光','点击']].mean().reset_index()

candidates = []
for act in activities:
    for day in days:
        unique_combos = df[['渠道','广告系列ID_h','广告组ID_h']].drop_duplicates().values
        for ch, se, gr in unique_combos:
            mask = (grp_mean['活动']==act)&(grp_mean['渠道']==ch)&(grp_mean['广告系列ID_h']==se)&(grp_mean['广告组ID_h']==gr)
            row = grp_mean[mask]
            if not row.empty:
                spend = float(row['花费'].iloc[0])
                expo = float(row['曝光'].iloc[0])
                click = float(row['点击'].iloc[0])
            else:
                mask2 = (grp_mean['渠道']==ch)&(grp_mean['广告系列ID_h']==se)&(grp_mean['广告组ID_h']==gr)
                row2 = grp_mean[mask2]
                if not row2.empty:
                    spend = float(row2['花费'].iloc[0])
                    expo = float(row2['曝光'].iloc[0])
                    click = float(row2['点击'].iloc[0])
                else:
                    spend = 100.0
                    expo = 1000.0
                    click = 10.0
            candidates.append({
                '活动': act,
                'day': int(day),
                '渠道': ch,
                '广告系列ID_h': se,
                '广告组ID_h': gr,
                '花费': spend,
                '曝光': expo,
                '点击': click
            })
candidates_df = pd.DataFrame(candidates)

for col in ['花费','曝光','点击']:
    candidates_df[col] = pd.to_numeric(candidates_df[col], errors='coerce').fillna(0.0)
    candidates_df[col] = candidates_df[col].replace([np.inf, -np.inf], 0.0)

# ---------------- 4. 预测订单 ----------------
pred_feat_df = candidates_df[['花费','曝光','点击','渠道','广告系列ID_h','广告组ID_h','day']].rename(columns={'day':'活动第几天'})
pred_X_enc = pd.get_dummies(pred_feat_df, columns=['渠道','广告系列ID_h','广告组ID_h'], drop_first=False).astype(float)

for col in X_enc.columns:
    if col not in pred_X_enc.columns:
        pred_X_enc[col] = 0.0
pred_X_enc = pred_X_enc[X_enc.columns]

candidates_df['predicted_orders'] = rf.predict(pred_X_enc)
candidates_df['predicted_orders'] = pd.to_numeric(candidates_df['predicted_orders'], errors='coerce').fillna(0.0).clip(lower=0.0)

# ---------------- 5. 节假日加权 ----------------
candidates_df['date'] = pd.to_datetime(MONTH + '-' + candidates_df['day'].astype(str), errors='coerce')
candidates_df['is_holiday'] = candidates_df['date'].astype(str).isin(HOLIDAYS)
candidates_df['predicted_orders'] *= np.where(candidates_df['is_holiday'], HOLIDAY_MULTIPLIER, 1.0)

# ---------------- 6. 可选：裁剪候选 ----------------
if MAX_CANDIDATES is not None:
    top_k = MAX_CANDIDATES
    candidates_df = candidates_df.sort_values('predicted_orders', ascending=False).head(top_k).reset_index(drop=True)

# ---------------- 7. 多臂老虎机探索扰动 ----------------
np.random.seed(42)
noise = np.random.uniform(1 - EXPLORATION_STEP, 1 + EXPLORATION_STEP, size=len(candidates_df))
candidates_df['predicted_orders_explore'] = candidates_df['predicted_orders'] * noise
candidates_df['predicted_orders_explore'] = pd.to_numeric(candidates_df['predicted_orders_explore'], errors='coerce').fillna(0.0).clip(lower=0.0)

# ---------------- 8. LP 求解 ----------------
prob = LpProblem("Ad_Optimization_DayLevel", LpMaximize)
decision_vars = {i: LpVariable(f"x_{i}", cat=LpBinary) for i in candidates_df.index}

activity_day_pairs = candidates_df[['活动','day']].drop_duplicates().values.tolist()
day_var = {}
for act, d in activity_day_pairs:
    day_var[(act, int(d))] = LpVariable(f"day_{str(act)}_{int(d)}", cat=LpBinary)

prob += lpSum([decision_vars[i] * float(candidates_df.loc[i, 'predicted_orders_explore']) for i in candidates_df.index])

unique_channels = candidates_df['渠道'].nunique()
unique_series = candidates_df['广告系列ID_h'].nunique()
unique_groups = candidates_df['广告组ID_h'].nunique()
M = (unique_channels * unique_series * unique_groups) + 1

for (act, d) in activity_day_pairs:
    idxs = candidates_df[(candidates_df['活动']==act) & (candidates_df['day']==d)].index.tolist()
    if not idxs:
        continue
    prob += lpSum([decision_vars[i] for i in idxs]) >= day_var[(act, d)]
    prob += lpSum([decision_vars[i] for i in idxs]) <= M * day_var[(act, d)]

for act in candidates_df['活动'].unique():
    related_day_vars = [day_var[(a, d)] for (a, d) in day_var.keys() if a == act]
    if related_day_vars:
        prob += lpSum(related_day_vars) <= MAX_DAYS_PER_ACTIVITY

candidates_df['花费_for_lp'] = pd.to_numeric(candidates_df['花费'], errors='coerce').fillna(100.0)
candidates_df['花费_for_lp'] = candidates_df['花费_for_lp'].replace([np.inf, -np.inf], 100.0)
prob += lpSum([decision_vars[i] * float(candidates_df.loc[i,'花费_for_lp']) for i in candidates_df.index]) <= TOTAL_BUDGET

prob.solve()

# ---------------- 9. 输出结果 ----------------
candidates_df['selected'] = [float(decision_vars[i].varValue) if i in decision_vars else 0.0 for i in candidates_df.index]
final_plan = candidates_df[candidates_df['selected'] > 0.5].copy()
final_plan = final_plan.sort_values(['day','活动','渠道','广告系列ID_h','广告组ID_h']).reset_index(drop=True)

days_per_activity = final_plan.groupby('活动')['day'].nunique().rename('投放天数').reset_index()
combos_per_activity = final_plan.groupby('活动').size().rename('选中组合数').reset_index()

print("每个活动被安排的投放天数：")
print(days_per_activity)
print("\n每个活动的选中组合数：")
print(combos_per_activity)
print("\n输出示例（前30条）：")
print(final_plan[['day','date','活动','渠道','广告系列ID_h','广告组ID_h','花费_for_lp','predicted_orders','predicted_orders_explore']].head(30))

final_plan.to_excel(OUTPUT_PATH, index=False)
print(f"\n已保存最终安排到：{OUTPUT_PATH}")


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/zb/sl5gnty53p56nw3lwm225xcw0000gn/T/34365fb4ca2f40648bef28f906dd150a-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/zb/sl5gnty53p56nw3lwm225xcw0000gn/T/34365fb4ca2f40648bef28f906dd150a-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 321 COLUMNS
At line 18477 RHS
At line 18794 BOUNDS
At line 21950 ENDATA
Problem MODEL has 316 rows, 3155 columns and 8845 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 2.20014e+06 - 0.01 seconds
Cgl0004I processed model has 316 rows, 3155 columns (3155 integer (3155 of which binary)) and 8845 elements
Cbc0038I Initial state - 161 integers unsatisfied sum - 31.7992
Cbc0038I Pass   1: suminf.   31.10574 (159) obj. -1.7593e+06 itera

In [6]:
# -*- coding: utf-8 -*-
"""
完整广告预算分配脚本（修正 day-level 约束，含多臂老虎机探索）
- 路径：'/Users/qiulianpeng/比赛/赛题数据(天)_组别211.xlsx'
- 输出：'/Users/qiulianpeng/比赛/12月广告最终安排_fixed.xlsx'
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from pulp import LpProblem, LpVariable, lpSum, LpMaximize, LpBinary
from itertools import product
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

# ---------------- 参数 ----------------
FILE_PATH = '/Users/qiulianpeng/比赛/赛题数据(天)_组别211.xlsx'
MONTH = '2026-12'
HOLIDAYS = ['2026-12-20', '2026-12-22', '2026-12-24', '2026-12-25']
HOLIDAY_MULTIPLIER = 1              # 节假日效益倍数
TOTAL_BUDGET = 1_000_000            # 总预算
MAX_DAYS_PER_ACTIVITY = 6           # 每个活动最多投放天数（不同日期数）
EXPLORATION_STEP = 0.1              # 探索扰动幅度（±）
MAX_CANDIDATES = 3000               # 可选：若候选过多，设置为 int 保留 top-K（按预测订单）；None 表示不裁剪

OUTPUT_PATH = '/Users/qiulianpeng/比赛/12月广告最终安排_fixed-mab-无节假日版.xlsx'

# ---------------- 1. 读取历史数据 ----------------
df = pd.read_excel(FILE_PATH)
df.columns = df.columns.str.strip()

required_cols = ['活动','活动第几天','渠道','广告系列ID_h','广告组ID_h','业绩','订单','花费','曝光','点击']
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"历史数据缺少列：{c}")

num_cols = ['业绩','订单','花费','曝光','点击']
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# ---------------- 2. 准备训练特征 ----------------
train_features = ['渠道','广告系列ID_h','广告组ID_h','花费','曝光','点击','活动第几天']
target = '订单'

for c in ['渠道','广告系列ID_h','广告组ID_h']:
    df[c] = df[c].astype('category')

X = df[train_features]
y = df[target].astype(float)

X_enc = pd.get_dummies(X, columns=['渠道','广告系列ID_h','广告组ID_h'], drop_first=False).astype(float)
X_train, X_val, y_train, y_val = train_test_split(X_enc, y, test_size=0.2, random_state=42)

# 使用随机森林替换 LightGBM
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# ---------------- 3. 生成候选组合 ----------------
activities = df['活动'].unique()
channels = df['渠道'].cat.categories if hasattr(df['渠道'], 'cat') else df['渠道'].unique()
series = df['广告系列ID_h'].cat.categories if hasattr(df['广告系列ID_h'], 'cat') else df['广告系列ID_h'].unique()
groups = df['广告组ID_h'].cat.categories if hasattr(df['广告组ID_h'], 'cat') else df['广告组ID_h'].unique()
days = list(range(1, 32))

grp_mean = df.groupby(['活动','渠道','广告系列ID_h','广告组ID_h'])[['花费','曝光','点击']].mean().reset_index()

candidates = []
for act in activities:
    for day in days:
        unique_combos = df[['渠道','广告系列ID_h','广告组ID_h']].drop_duplicates().values
        for ch, se, gr in unique_combos:
            mask = (grp_mean['活动']==act)&(grp_mean['渠道']==ch)&(grp_mean['广告系列ID_h']==se)&(grp_mean['广告组ID_h']==gr)
            row = grp_mean[mask]
            if not row.empty:
                spend = float(row['花费'].iloc[0])
                expo = float(row['曝光'].iloc[0])
                click = float(row['点击'].iloc[0])
            else:
                mask2 = (grp_mean['渠道']==ch)&(grp_mean['广告系列ID_h']==se)&(grp_mean['广告组ID_h']==gr)
                row2 = grp_mean[mask2]
                if not row2.empty:
                    spend = float(row2['花费'].iloc[0])
                    expo = float(row2['曝光'].iloc[0])
                    click = float(row2['点击'].iloc[0])
                else:
                    spend = 100.0
                    expo = 1000.0
                    click = 10.0
            candidates.append({
                '活动': act,
                'day': int(day),
                '渠道': ch,
                '广告系列ID_h': se,
                '广告组ID_h': gr,
                '花费': spend,
                '曝光': expo,
                '点击': click
            })
candidates_df = pd.DataFrame(candidates)

for col in ['花费','曝光','点击']:
    candidates_df[col] = pd.to_numeric(candidates_df[col], errors='coerce').fillna(0.0)
    candidates_df[col] = candidates_df[col].replace([np.inf, -np.inf], 0.0)

# ---------------- 4. 预测订单 ----------------
pred_feat_df = candidates_df[['花费','曝光','点击','渠道','广告系列ID_h','广告组ID_h','day']].rename(columns={'day':'活动第几天'})
pred_X_enc = pd.get_dummies(pred_feat_df, columns=['渠道','广告系列ID_h','广告组ID_h'], drop_first=False).astype(float)

for col in X_enc.columns:
    if col not in pred_X_enc.columns:
        pred_X_enc[col] = 0.0
pred_X_enc = pred_X_enc[X_enc.columns]

candidates_df['predicted_orders'] = rf.predict(pred_X_enc)
candidates_df['predicted_orders'] = pd.to_numeric(candidates_df['predicted_orders'], errors='coerce').fillna(0.0).clip(lower=0.0)

# ---------------- 5. 节假日加权 ----------------
candidates_df['date'] = pd.to_datetime(MONTH + '-' + candidates_df['day'].astype(str), errors='coerce')
candidates_df['is_holiday'] = candidates_df['date'].astype(str).isin(HOLIDAYS)
candidates_df['predicted_orders'] *= np.where(candidates_df['is_holiday'], HOLIDAY_MULTIPLIER, 1.0)

# ---------------- 6. 可选：裁剪候选 ----------------
if MAX_CANDIDATES is not None:
    top_k = MAX_CANDIDATES
    candidates_df = candidates_df.sort_values('predicted_orders', ascending=False).head(top_k).reset_index(drop=True)

# ---------------- 7. 多臂老虎机探索扰动（升级为 Thompson Sampling） ----------------
# 为了不删除原代码的均匀扰动实现，我保留原有 noise 的生成（不改变其它流程）：
np.random.seed(42)
noise = np.random.uniform(1 - EXPLORATION_STEP, 1 + EXPLORATION_STEP, size=len(candidates_df))
candidates_df['predicted_orders_explore'] = candidates_df['predicted_orders'] * noise
candidates_df['predicted_orders_explore'] = pd.to_numeric(candidates_df['predicted_orders_explore'], errors='coerce').fillna(0.0).clip(lower=0.0)

# ----------------- Thompson Sampling 实现（真正的 MAB 部分） -----------------
# 定义 arm：我们以 (活动, 渠道) 作为一个 arm（粒度可扩展）
arms = [ (act, ch) for act in candidates_df['活动'].unique() for ch in candidates_df['渠道'].unique() ]
# 初始化 Beta 参数（alpha, beta），使用字典保存
alpha = defaultdict(lambda: 1.0)
beta = defaultdict(lambda: 1.0)

# 为每个 candidate 依据其所属 arm 采样 theta，再把 theta 映射到 [1-EXPLORATION_STEP, 1+EXPLORATION_STEP] 的 noise 范围
# mapping: noise = (1 - EXPLORATION_STEP) + theta * (2 * EXPLORATION_STEP)
# 这样 theta=0 -> 1-EXPLORATION_STEP, theta=1 -> 1+EXPLORATION_STEP
theta_map = {}
for arm in arms:
    theta_map[arm] = np.random.beta(alpha[arm], beta[arm])

# 为每个候选项生成对应 arm 的采样 noise（覆盖全部候选）
ts_noise = []
for idx, row in candidates_df.iterrows():
    arm = (row['活动'], row['渠道'])
    # 如果 arm 未在 theta_map（罕见），则直接从 Beta(alpha,beta) 采样
    if arm not in theta_map:
        theta = np.random.beta(alpha[arm], beta[arm])
    else:
        theta = np.random.beta(alpha[arm], beta[arm])  # 每次运行都重新采样 (Thompson)
    noise_ts = (1 - EXPLORATION_STEP) + theta * (2 * EXPLORATION_STEP)
    ts_noise.append(noise_ts)

# 用 Thompson Sampling 的 noise 覆盖之前的 predicted_orders_explore，完成真正的 MAB 探索扰动
candidates_df['predicted_orders_explore'] = candidates_df['predicted_orders'] * np.array(ts_noise)
candidates_df['predicted_orders_explore'] = pd.to_numeric(candidates_df['predicted_orders_explore'], errors='coerce').fillna(0.0).clip(lower=0.0)

# 为后续更新做准备：记录每个 candidate 的 arm 信息
candidates_df['arm'] = list(zip(candidates_df['活动'], candidates_df['渠道']))

# ---------------- 8. LP 求解 ----------------
prob = LpProblem("Ad_Optimization_DayLevel", LpMaximize)
decision_vars = {i: LpVariable(f"x_{i}", cat=LpBinary) for i in candidates_df.index}

activity_day_pairs = candidates_df[['活动','day']].drop_duplicates().values.tolist()
day_var = {}
for act, d in activity_day_pairs:
    day_var[(act, int(d))] = LpVariable(f"day_{str(act)}_{int(d)}", cat=LpBinary)

prob += lpSum([decision_vars[i] * float(candidates_df.loc[i, 'predicted_orders_explore']) for i in candidates_df.index])

unique_channels = candidates_df['渠道'].nunique()
unique_series = candidates_df['广告系列ID_h'].nunique()
unique_groups = candidates_df['广告组ID_h'].nunique()
M = (unique_channels * unique_series * unique_groups) + 1

for (act, d) in activity_day_pairs:
    idxs = candidates_df[(candidates_df['活动']==act) & (candidates_df['day']==d)].index.tolist()
    if not idxs:
        continue
    prob += lpSum([decision_vars[i] for i in idxs]) >= day_var[(act, d)]
    prob += lpSum([decision_vars[i] for i in idxs]) <= M * day_var[(act, d)]

for act in candidates_df['活动'].unique():
    related_day_vars = [day_var[(a, d)] for (a, d) in day_var.keys() if a == act]
    if related_day_vars:
        prob += lpSum(related_day_vars) <= MAX_DAYS_PER_ACTIVITY

candidates_df['花费_for_lp'] = pd.to_numeric(candidates_df['花费'], errors='coerce').fillna(100.0)
candidates_df['花费_for_lp'] = candidates_df['花费_for_lp'].replace([np.inf, -np.inf], 100.0)
prob += lpSum([decision_vars[i] * float(candidates_df.loc[i,'花费_for_lp']) for i in candidates_df.index]) <= TOTAL_BUDGET

prob.solve()

# ---------------- 9. 输出结果 ----------------
candidates_df['selected'] = [float(decision_vars[i].varValue) if i in decision_vars else 0.0 for i in candidates_df.index]
final_plan = candidates_df[candidates_df['selected'] > 0.5].copy()
final_plan = final_plan.sort_values(['day','活动','渠道','广告系列ID_h','广告组ID_h']).reset_index(drop=True)

days_per_activity = final_plan.groupby('活动')['day'].nunique().rename('投放天数').reset_index()
combos_per_activity = final_plan.groupby('活动').size().rename('选中组合数').reset_index()

print("每个活动被安排的投放天数：")
print(days_per_activity)
print("\n每个活动的选中组合数：")
print(combos_per_activity)
print("\n输出示例（前30条）：")
print(final_plan[['day','date','活动','渠道','广告系列ID_h','广告组ID_h','花费_for_lp','predicted_orders','predicted_orders_explore']].head(30))

# ==== MAB 参数更新（用被选中的候选项估计 reward，并更新 alpha/beta） ====
# 这里用一个 proxy reward：candidate 的 predicted_orders 相对标准化值（归一化到 0-1）作为成功概率近似
eps = 1e-9
max_pred = candidates_df['predicted_orders'].max() + eps
# 对每个被选中的 candidate，用其 predicted_orders / max_pred 作为 reward 更新对应 arm
for idx, row in candidates_df[candidates_df['selected'] > 0.5].iterrows():
    arm = row['arm']
    # reward in [0,1]
    reward = float(row['predicted_orders'] / max_pred)
    # 把 reward 当作成功概率的估计进行累加更新（alpha 加 reward，beta 加 1-reward）
    alpha[arm] += reward
    beta[arm] += (1.0 - reward)

# 保存 MAB alpha/beta 到 candidates_df（便于后续分析/可视化）
# map alpha/beta 到每 candidate 的 arm
candidates_df['arm_alpha'] = candidates_df['arm'].apply(lambda a: alpha[a])
candidates_df['arm_beta'] = candidates_df['arm'].apply(lambda a: beta[a])

final_plan.to_excel(OUTPUT_PATH, index=False)
print(f"\n已保存最终安排到：{OUTPUT_PATH}")


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/zb/sl5gnty53p56nw3lwm225xcw0000gn/T/3dcfd28ef38a4ad49b099e3e930c8e5d-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/zb/sl5gnty53p56nw3lwm225xcw0000gn/T/3dcfd28ef38a4ad49b099e3e930c8e5d-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 321 COLUMNS
At line 18477 RHS
At line 18794 BOUNDS
At line 21950 ENDATA
Problem MODEL has 316 rows, 3155 columns and 8845 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 2.21474e+06 - 0.01 seconds
Cgl0004I processed model has 316 rows, 3155 columns (3155 integer (3155 of which binary)) and 8845 elements
Cbc0038I Initial state - 161 integers unsatisfied sum - 31.1056
Cbc0038I Pass   1: suminf.   30.45165 (159) obj. -1.77283e+06 iter